# Real user-story test for the PROMISE-trained classifiers

Goal: close the biggest gap in the paper: the framework was never tested on user stories.

Steps: (1) get public user-story sets, (2) sample about 250 stories across projects, (3) two annotators label them F / NFR / FQ,
(4) Cohen's kappa, (5) evaluate PROMISE-trained models on the stories, with project-level bootstrap CIs.

**Data (verify before citing):** the public user-story collection of Dalpiaz (about 22 sets, one story per line, hosted on Mendeley Data).
Search "Dalpiaz user stories dataset Mendeley", check the licence and citation, and upload the `.txt` files into `data/user_stories/`.

In [ ]:
!pip -q install transformers scikit-learn scipy pandas matplotlib liac-arff accelerate

In [ ]:
import os, glob, re, warnings
warnings.filterwarnings('ignore')
SEEDS = [42, 43, 44]
N_STORIES = 250
os.makedirs('data/user_stories', exist_ok=True)
print(sorted(glob.glob('data/user_stories/*.txt')))

In [ ]:
import os, urllib.request, arff, numpy as np, pandas as pd
os.makedirs('data', exist_ok=True)
URL = 'https://zenodo.org/records/268542/files/nfr.arff?download=1'
PATH = 'data/nfr.arff'
if not os.path.exists(PATH):
    try:
        urllib.request.urlretrieve(URL, PATH)
    except Exception as e:
        print('Download failed. Upload nfr.arff into the data/ folder by hand.', e)
d = arff.load(open(PATH))
cols = [a[0] for a in d['attributes']]
df = pd.DataFrame(d['data'], columns=cols)
pc = [c for c in cols if 'project' in c.lower()][0]
tc = [c for c in cols if 'text' in c.lower()][0]
cc = [c for c in cols if c.lower().startswith('class')][0]
df = df.rename(columns={pc: 'project', tc: 'text', cc: 'cls'})[['project', 'text', 'cls']]
df['cls'] = df['cls'].astype(str).str.strip()
df['project'] = df['project'].astype(str)
assert len(df) == 625, f'Expected 625 requirements, got {len(df)}'
print(df['cls'].value_counts().to_dict(), '| projects:', df['project'].nunique())

In [ ]:
# Load stories, keep only well-formed Connextra stories, sample evenly across projects.
PAT = re.compile(r"^\s*as\s+(?:an?|the)\s+(?P<role>.+?),?\s+i\s+(?:want|need|can|would like|am able)\s+(?:to\s+)?(?P<goal>.+?)(?:,?\s+(?:so that|in order to|so)\s+(?P<ben>.+))?\s*$", re.I | re.S)
rows = []
for f in sorted(glob.glob('data/user_stories/*.txt')):
    proj = os.path.splitext(os.path.basename(f))[0]
    for i, line in enumerate(open(f, encoding='utf-8', errors='ignore')):
        line = line.strip()
        m = PAT.match(line)
        if m:
            rows.append(dict(project=proj, story=line, goal=m.group('goal').strip(), benefit=(m.group('ben') or '').strip()))
us = pd.DataFrame(rows).drop_duplicates('story')
print(len(us), 'parsed stories from', us['project'].nunique(), 'projects')
assert len(us) > 0, 'Upload the user-story .txt files first'
per = max(1, N_STORIES // us['project'].nunique())
sample = (us.groupby('project', group_keys=False).apply(lambda x: x.sample(min(len(x), per), random_state=42))
            .sample(frac=1, random_state=1).reset_index(drop=True))
sample.insert(0, 'id', range(len(sample)))
print(len(sample), 'sampled')

## Labelling guideline (give this to both annotators; label the story text, not what you imagine around it)
- **F**: the story asks only for a behaviour or capability (what the system does).
- **NFR**: the story is mainly a quality constraint (speed, security, usability, availability, look and feel, legal, maintainability...).
- **FQ**: the story asks for behaviour and also states or clearly implies a quality aspect (e.g. "log failed logins with time and IP so I can spot attacks").
- Decide independently. Do not discuss until both have finished. Write a short note for doubtful items.
- Annotator 2 should label without seeing Annotator 1's column. Use a classmate or friend with RE or SE background; record who and their experience.

In [ ]:
# Export sheets for the two annotators (no labels visible to each other)
for a in ['A1', 'A2']:
    s = sample[['id', 'project', 'story']].copy(); s['label'] = ''; s['note'] = ''
    s.to_csv(f'annotation_{a}.csv', index=False)
sample.to_csv('sample_full.csv', index=False)
print('Send annotation_A1.csv / annotation_A2.csv to the annotators. Fill the "label" column with F, NFR or FQ.')
try:
    from google.colab import files
    files.download('annotation_A1.csv'); files.download('annotation_A2.csv')
except Exception:
    pass

## After labelling
Upload the two completed CSVs as `annotation_A1_done.csv` and `annotation_A2_done.csv`, then run the cells below.

In [ ]:
from sklearn.metrics import cohen_kappa_score, confusion_matrix
a1 = pd.read_csv('annotation_A1_done.csv'); a2 = pd.read_csv('annotation_A2_done.csv')
m = sample.merge(a1[['id', 'label']].rename(columns={'label': 'l1'}), on='id').merge(a2[['id', 'label']].rename(columns={'label': 'l2'}), on='id')
for c in ['l1', 'l2']:
    m[c] = m[c].astype(str).str.strip().str.upper().replace({'F+Q': 'FQ', 'NF': 'NFR', 'NONFUNCTIONAL': 'NFR'})
assert set(m['l1']) <= {'F', 'NFR', 'FQ'} and set(m['l2']) <= {'F', 'NFR', 'FQ'}, 'Unexpected labels'
print('Cohen kappa (3 classes):', round(cohen_kappa_score(m['l1'], m['l2']), 3))
print('Raw agreement:', round((m['l1'] == m['l2']).mean(), 3))
print(pd.crosstab(m['l1'], m['l2']))
dis = m[m['l1'] != m['l2']][['id', 'story', 'l1', 'l2']]
dis.to_csv('disagreements.csv', index=False)
print(len(dis), 'disagreements -> resolve by discussion or a third annotator, then fill gold.')

In [ ]:
# Adjudicated gold labels: edit disagreements_resolved.csv (columns id, gold) and upload it. Agreed items keep their label.
if os.path.exists('disagreements_resolved.csv'):
    res = pd.read_csv('disagreements_resolved.csv').set_index('id')['gold']
    m['gold'] = [res.get(i, a) if a != b else a for i, a, b in zip(m['id'], m['l1'], m['l2'])]
else:
    print('No disagreements_resolved.csv yet: using Annotator 1 as a TEMPORARY gold. Do not report that.')
    m['gold'] = m['l1']
print(m['gold'].value_counts().to_dict())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

def make_baseline(name):
    vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, stop_words='english')
    clf = {'nb': MultinomialNB(alpha=0.1),
           'svm': LinearSVC(C=1, class_weight='balanced'),
           'lr': LogisticRegression(C=10, class_weight='balanced', max_iter=2000)}[name]
    return make_pipeline(vec, clf)

def baseline_scores(name, X_tr, y_tr, X_te, K):
    pipe = make_baseline(name).fit(X_tr, y_tr)
    s = pipe.predict_proba(X_te) if hasattr(pipe, 'predict_proba') else pipe.decision_function(X_te)
    if s.ndim == 1:
        s = np.c_[-s, s]
    out = np.full((len(X_te), K), -1e9)
    out[:, pipe[-1].classes_] = s
    return out

In [ ]:
import torch, random
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def train_transformer(model_name, X_tr, y_tr, X_te, K, seed, lr=3e-5, epochs=5, bs=16, maxlen=64):
    """Full fine-tuning (no frozen layers), class-weighted loss as in Eq. (1). Returns test logits."""
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=K).to(DEVICE)
    counts = np.bincount(y_tr, minlength=K)
    w = len(y_tr) / (K * np.maximum(counts, 1))
    lossf = torch.nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float, device=DEVICE))
    enc = lambda t: tok(list(t), truncation=True, max_length=maxlen, padding=True, return_tensors='pt')
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = epochs * int(np.ceil(len(X_tr) / bs))
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    g = torch.Generator().manual_seed(seed)
    use_cuda = DEVICE == 'cuda'
    scaler = torch.cuda.amp.GradScaler(enabled=use_cuda)
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(X_tr), generator=g).numpy()
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            e = {k: v.to(DEVICE) for k, v in enc(X_tr[b]).items()}
            yb = torch.tensor(y_tr[b], device=DEVICE)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_cuda):
                out = model(**e).logits
            loss = lossf(out.float(), yb)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step()
    model.eval()
    outs = []
    with torch.no_grad():
        for i in range(0, len(X_te), 64):
            e = {k: v.to(DEVICE) for k, v in enc(X_te[i:i + 64]).items()}
            outs.append(model(**e).logits.float().cpu().numpy())
    del model
    if use_cuda:
        torch.cuda.empty_cache()
    return np.concatenate(outs)

In [ ]:
# Train on all PROMISE (Task A: FR vs NFR), predict the stories. Text = goal + benefit (the PROMISE-like part).
Xp = df['text'].values; yp = (df['cls'] != 'F').astype(int).values
Xs = (m['goal'] + ' ' + m['benefit']).str.strip().values
SC = {}
for s in SEEDS:
    SC[('svm', s)] = baseline_scores('svm', Xp, yp, Xs, 2)
    SC[('distilbert', s)] = train_transformer('distilbert-base-uncased', Xp, yp, Xs, 2, s, lr=5e-5)
    SC[('roberta', s)] = train_transformer('roberta-base', Xp, yp, Xs, 2, s, lr=3e-5)
    print('done seed', s)

In [ ]:
from sklearn.metrics import f1_score, precision_recall_fscore_support
gold = m['gold'].values; proj = m['project'].values
binary = np.isin(gold, ['F', 'NFR']); yb = (gold == 'NFR').astype(int)
qgold = np.isin(gold, ['NFR', 'FQ']).astype(int)      # any quality aspect present

def metrics(scores):
    pred = scores.argmax(1)
    f_bin = f1_score(yb[binary], pred[binary], average='macro', zero_division=0)
    p, r, f, _ = precision_recall_fscore_support(qgold, pred, average='binary', zero_division=0)  # pred NFR = flags a quality concern
    fq_flag = pred[gold == 'FQ'].mean() if (gold == 'FQ').any() else np.nan
    return f_bin, p, r, f, fq_flag

out = []
for model in ['svm', 'distilbert', 'roberta']:
    ms = np.array([metrics(SC[(model, s)]) for s in SEEDS])
    mu = ms.mean(0)
    out.append(dict(model=model, macroF1_F_vs_NFR_only=mu[0], quality_flag_P=mu[1], quality_flag_R=mu[2], quality_flag_F1=mu[3], share_FQ_flagged=mu[4]))
res_tab = pd.DataFrame(out).round(3); print(res_tab.to_string()); res_tab.to_csv('user_story_results.csv', index=False)
print('Gold distribution:', pd.Series(gold).value_counts().to_dict())
print('Note: models cannot output FQ. "share_FQ_flagged" = how often an FQ story is predicted NFR (its quality aspect noticed).')

In [ ]:
# Cluster bootstrap by source project (95% CI), seed-averaged; difference vs SVM on the binary macro-F1
rng = np.random.default_rng(0); B = 2000
projs = np.unique(proj); idx_by = {p: np.where(proj == p)[0] for p in projs}
def bf1(model, idx):
    v = []
    for s in SEEDS:
        pr = SC[(model, s)].argmax(1)[idx]; b = binary[idx]
        v.append(f1_score(yb[idx][b], pr[b], average='macro', zero_division=0) if b.sum() > 1 else np.nan)
    return np.nanmean(v)
draws = [np.concatenate([idx_by[p] for p in rng.choice(projs, len(projs), replace=True)]) for _ in range(B)]
rows = []
for model in ['svm', 'distilbert', 'roberta']:
    vals = [bf1(model, i) for i in draws]
    lo, hi = np.nanpercentile(vals, [2.5, 97.5]); rows.append(dict(model=model, macroF1=round(bf1(model, np.arange(len(gold))), 3), CI95=f'[{lo:.3f}, {hi:.3f}]'))
for model in ['distilbert', 'roberta']:
    d = [bf1(model, i) - bf1('svm', i) for i in draws]; lo, hi = np.nanpercentile(d, [2.5, 97.5])
    rows.append(dict(model=f'{model} minus svm', macroF1=round(bf1(model, np.arange(len(gold))) - bf1('svm', np.arange(len(gold))), 3), CI95=f'[{lo:.3f}, {hi:.3f}]'))
bt = pd.DataFrame(rows); print(bt.to_string()); bt.to_csv('user_story_bootstrap.csv', index=False)

In [ ]:
# Confidence gate on real stories (transformers): coverage vs precision of auto-labels (F vs NFR items only)
from scipy.special import softmax
import matplotlib.pyplot as plt
taus = np.linspace(0.5, 0.995, 40)
for model in ['distilbert', 'roberta']:
    P = []
    for s in SEEDS:
        p = softmax(SC[(model, s)], 1)[binary]; c, pr = p.max(1), p.argmax(1)
        P.append([((c >= t).mean(), (pr[c >= t] == yb[binary][c >= t]).mean() if (c >= t).any() else np.nan) for t in taus])
    P = np.nanmean(np.array(P), axis=0); plt.plot(P[:, 0], P[:, 1], label=model)
plt.axhline(0.95, ls='--', c='grey'); plt.xlabel('coverage'); plt.ylabel('precision'); plt.legend(); plt.title('Confidence gate on user stories')
plt.savefig('us_coverage_precision.png', dpi=200); plt.show()

## How to report this in the paper
- New subsection "Evaluation on real user stories": dataset source and licence, sampling (N, projects), annotators, guideline, Cohen's kappa, adjudication.
- Table: macro-F1 and quality-flag P/R/F1 for SVM, DistilBERT, RoBERTa with 95% project-level CIs.
- Say plainly that the models were trained on PROMISE "shall" statements and cannot predict FQ. Low scores are an honest, publishable finding about format shift.
- Do not claim the framework is validated in the field; this is still an offline test. Update Sections V-A, V-C (format shift), VI and the abstract.